# NB06 · Chemical Databases: PubChem, ChEMBL, ZINC, and Friends

**Part B: Data & Chemistry** · Estimated time: ~90 min · Difficulty: ●●○

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prateekpaul/Cheminformatics/blob/main/notebooks/nb06_chemical_databases_chembl_pubchem.ipynb)

---

### One-Line Summary
Explore the primary public repositories of chemical and bioactivity data, query ChEMBL programmatically with offline fallbacks, master the data curation and deduplication pipeline, inspect 3D conformational torsion preferences from crystallographic data, and profile library property distributions.

## 1. Learning Objectives

By the end of this notebook, you will be able to:
1. **Navigate the Database Atlas**: Distinguish the scope, size, content, and licensing of PubChem, ChEMBL, DrugBank, ZINC, PDB, CSD, and specialized metabolomics databases.
2. **Programmatically Retrieve Bioactivity Data**: Query the ChEMBL API for human Acetylcholinesterase (AChE) inhibitors, apply rigorous bioactivity filters, and implement offline caching.
3. **Inspect Database Records**: Parse and understand metadata in multi-molecule SDF files and metabolome CSV files.
4. **Execute the Standard Curation Pipeline**: Standardize molecules using `chemlib.standardize` (salt stripping, uncharging, canonical SMILES, tautomer handling).
5. **Aggregate Duplicates & Compute $\text{pIC}_{50}$**: Group identical structures, calculate median bioactivities and interquartile ranges (IQR), and transform molar concentrations into logarithmic $\text{pIC}_{50}$ scale.
6. **Mine Structural & Torsional Data**: Analyze CSD/PDB structural preferences (O–C–C–O gauche vs. C–C–C–C anti) and explain their stereoelectronic origin.
7. **Profile Library Physicochemical Distributions**: Compute Lipinski parameters (MW, LogP, HBD, HBA) and visualize drug-likeness distributions.

## 2. Prerequisites & Lecture Links

- **Prerequisites**: Familiarity with SMILES and InChI notation (`nb03`), molecular graph representations (`nb04`), and chemical file formats (`nb05`).
- **Lecture Aligned**: Lecture Deck 02 (*Structure Representation & Databases*), Lecture Deck 04 (*Public Domain Libraries*), and Lecture Deck 07 (*Small-Molecule Databases*).
- **Core Concepts**: Target ID resolution, bioactivity endpoints ($	ext{IC}_{50}, K_i, K_d$), salt stripping, duplicate consensus, gauche effect, Lipinski Rule of 5.

In [ ]:
# Setup and environment configuration
import os, sys, random, warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Draw, Descriptors

# Ensure src directory is in sys.path for chemlib imports
repo_root = Path.cwd().parent if Path.cwd().name in ["notebooks", "solutions"] else Path.cwd()
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

import chemlib
from chemlib import standardize, descriptors

SEED = 42
FULL_RUN = False          # Set True to execute live API queries against ChEMBL servers
random.seed(SEED)
np.random.seed(SEED)
RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore", category=DeprecationWarning)

sns.set_theme(style="whitegrid", font_scale=1.1)
print("Environment initialized. chemlib version:", chemlib.__version__)
# Course reproducibility seeds and execution mode
import random


## 3. The Chemical Database Atlas

Computational drug discovery relies on public and commercial databases spanning chemical structures, bioactivities, crystal structures, and metabolomics.

| Database | Primary Content | Size Order of Magnitude (2025/2026) | Access Method / API | License / Availability |
|:---|:---|:---|:---|:---|
| **PubChem** (NIH) | Small molecules, bioassays, patents, substance records | > 115 million compounds; > 300 million substances | PUG-REST, PUG-View, FTP dumps | Open Access (Public Domain) |
| **ChEMBL** (EMBL-EBI) | Curated bioactivity data mapped to drug targets | > 2.4 million compounds; > 20 million bioactivities | `chembl_webresource_client`, REST API, PostgreSQL dump | CC BY-SA 3.0 |
| **DrugBank** (OMx) | Approved, investigational, and experimental drugs with mechanisms | ~ 15,000 drug entries (~2,700 approved small molecules) | XML download, Web interface, commercial API | Free academic / Commercial license |
| **ZINC20 / ZINC22** (UCSF) | Commercially available screening compounds and tangibles | > 1.4 billion 3D molecules; fragment subsets | Tranche downloads, CartBlanche API | Free for research |
| **Protein Data Bank (PDB)** | Macromolecular 3D structures (proteins, nucleic acids, complexes) | > 225,000 crystal, Cryo-EM, and NMR entries | REST API, mmCIF / PDB format downloads | Public Domain (CC0) |
| **CSD** (CCDC) | Organic & organometallic small-molecule crystal structures | > 1.25 million curated X-ray crystal structures | CCDC Python API, ConQuest, Mercury | Academic subscription / Commercial |
| **YMDB / HMDB** (Wishart Lab) | Yeast / Human endogenous metabolites, pathways, NMR/MS spectra | ~ 2,000 (YMDB) / > 220,000 (HMDB) metabolites | XML / CSV / SDF dumps | Open Access |
| **FlavorDB** (Bagler Lab) | Flavor compounds, natural sources, sensory perception profiles | ~ 25,500 flavor molecules, 900+ ingredients | Web interface, RESTful endpoints | Free academic |
| **BindingDB** (UCSD) | Measured binding affinities ($K_i, K_d, 	ext{IC}_{50}$) for target-ligand pairs | ~ 2.8 million binding data points, 1.1M compounds | TSV download, REST API | CC BY 4.0 |

## 4. Inspecting Raw Database Files: SDF and YMDB CSV

Chemical databases distribute structures in standard interchange formats. Let's inspect:
1. An **SDFile (Structure-Data File)** containing 3D atomic coordinates and annotated SD tags.
2. A **Metabolome CSV File** from YMDB containing identifiers, formulas, SMILES, and InChIKeys.

In [ ]:
# Inspect the bundled SDF dataset (d03_fda_drugs.sdf)
sdf_path = repo_root / "data" / "processed" / "d03_fda_drugs.sdf"

if sdf_path.exists():
    suppl = Chem.SDMolSupplier(str(sdf_path))
    print(f"Total molecules in SDF: {len(suppl)}")
    
    # Inspect the first molecule
    mol_example = suppl[0]
    if mol_example is not None:
        name = mol_example.GetProp("_Name") if mol_example.HasProp("_Name") else "Unknown"
        print(f"\nFirst molecule: {name}")
        print(f"Heavy atom count: {mol_example.GetNumHeavyAtoms()}")
        print(f"Total atom count (with 3D explicit H): {mol_example.GetNumAtoms()}")
        print(f"Has 3D coordinates: {mol_example.GetConformer().Is3D()}")
        print("SD tags present:", list(mol_example.GetPropNames()))
else:
    print(f"File not found: {sdf_path}")

In [ ]:
# Inspect the bundled metabolome dataset (d08_ymdb_sample.csv)
ymdb_path = repo_root / "data" / "processed" / "d08_ymdb_sample.csv"
df_ymdb = pd.read_csv(ymdb_path)
print(f"Loaded {len(df_ymdb)} YMDB metabolite records:")
df_ymdb.head()

## 5. Programmatic Bioactivity Retrieval from ChEMBL

In modern drug discovery, retrieving bioactivity datasets from **ChEMBL** via its official client (`chembl_webresource_client`) is an essential workflow.

### Target of Interest: Human Acetylcholinesterase (AChE)
- **UniProt**: `P22303`
- **Target ChEMBL ID**: `CHEMBL220`
- **Therapeutic Role**: Target for symptomatic treatment of Alzheimer's disease (e.g., donepezil, rivastigmine, galantamine).

When querying bioactivity databases, we must specify:
1. **Assay target**: `CHEMBL220`
2. **Endpoint type**: `standard_type == 'IC50'` (half-maximal inhibitory concentration)
3. **Relationship**: `standard_relation == '='` (exact concentration; exclude open-ended `>` or `<` bounds)
4. **Units**: `standard_units == 'nM'`

In [ ]:
# Function to retrieve ChEMBL bioactivity with robust offline fallback
def fetch_chembl_target_activities(target_chembl_id="CHEMBL220", activity_type="IC50", use_cache=True):
    cached_file = repo_root / "data" / "processed" / "d04_chembl_ache_curated.csv"
    
    if use_cache or not FULL_RUN:
        print(f"Loading curated offline cache from {cached_file.name}...")
        return pd.read_csv(cached_file)
    
    try:
        from chembl_webresource_client.new_client import new_client
        print(f"Querying live ChEMBL API for {target_chembl_id} ({activity_type})...")
        activity_api = new_client.activity
        res = activity_api.filter(
            target_chembl_id=target_chembl_id,
            standard_type=activity_type,
            standard_units="nM",
            standard_relation="="
        ).only([
            "molecule_chembl_id", "canonical_smiles", "standard_type",
            "standard_relation", "standard_value", "standard_units",
            "target_chembl_id", "target_name"
        ])
        df = pd.DataFrame(list(res[:500]))
        return df
    except Exception as e:
        print(f"Live query unavailable ({e}); falling back to offline cache.")
        return pd.read_csv(cached_file)

df_raw_ache = fetch_chembl_target_activities("CHEMBL220", "IC50", use_cache=True)
print(f"Retrieved {len(df_raw_ache)} raw bioactivity records.")
df_raw_ache.head()

## 6. The Molecular Curation & Standardization Pipeline

Raw chemical datasets from literature-mined databases like ChEMBL contain counterions, solvent adducts, protonation discrepancies, and duplicate assays from multiple laboratories. A rigorous curation pipeline is mandatory before descriptor calculation or QSAR modeling.

```
Raw Data (SMILES + IC50)
   │
   ▼
1. Validity Check ───────► Drop unparseable SMILES & non-positive IC50
   │
   ▼
2. Salt Stripping ───────► Retain largest organic fragment (remove HCl, Na+, TFA)
   │
   ▼
3. Neutralization ───────► Standardize formal charges (uncharge carboxylates, amines)
   │
   ▼
4. Canonicalization ─────► Canonical SMILES representation
   │
   ▼
5. Duplicate Handling ───► Aggregate identical SMILES (median IC50; flag high IQR)
   │
   ▼
6. Activity Transform ───► Compute pIC50 = -log10(IC50 [M])
   │
   ▼
7. Binary Labelling ─────► Active (pIC50 >= 6.0) vs Inactive (pIC50 <= 5.0)
```

In [ ]:
# Demonstration of salt stripping and charge neutralization on individual test molecules
test_salt = "CC(=O)[O-].[Na+]"               # Sodium acetate
test_amine_salt = "CN(C)CCC.Cl"             # Triethylamine hydrochloride
test_zwitterion = "C(C(=O)[O-])[NH3+]"       # Glycine zwitterion

print("--- Individual Standardization Steps ---")
for smi in [test_salt, test_amine_salt, test_zwitterion]:
    m = Chem.MolFromSmiles(smi)
    stripped = standardize.remove_salts(m)
    neutralized = standardize.neutralize_charges(stripped)
    canonical_smi = Chem.MolToSmiles(neutralized)
    print(f"Raw: {smi:22} -> Stripped: {Chem.MolToSmiles(stripped):14} -> Clean: {canonical_smi}")

### Activity Transformation: Converting $\text{IC}_{50}$ to $\text{pIC}_{50}$

Biochemical assays report $\text{IC}_{50}$ in nanomolar ($	ext{nM}$). However, concentration response follows an exponential law. To linearize the free energy relationship $\Delta G^\circ = -RT \ln K_i$, we convert values to **$\text{pIC}_{50}$**:

$$\text{pIC}_{50} = -\log_{10}\left(\text{IC}_{50} \text{ [M]}\right) = -\log_{10}\left(\text{IC}_{50} \text{ [nM]} \times 10^{-9}\right) = 9 - \log_{10}\left(\text{IC}_{50} \text{ [nM]}\right)$$

- An $\text{IC}_{50}$ of $10\,\text{nM} \implies \text{pIC}_{50} = 9 - \log_{10}(10) = 8.0$ (potent).
- An $\text{IC}_{50}$ of $1,000\,\text{nM} \,(1\,\mu\text{M}) \implies \text{pIC}_{50} = 9 - 3 = 6.0$ (standard threshold for "active").
- An $\text{IC}_{50}$ of $10,000\,\text{nM} \,(10\,\mu\text{M}) \implies \text{pIC}_{50} = 9 - 4 = 5.0$ (threshold for "inactive").

In [ ]:
# Curate the raw AChE dataset using chemlib.standardize.curate_bioactivity_dataframe
curated_ache = standardize.curate_bioactivity_dataframe(
    df_raw_ache,
    smiles_col="canonical_smiles",
    activity_col="standard_value",
    expected_activity_type="IC50",
    active_threshold_nm=1000.0,
    inactive_threshold_nm=10000.0
)

print(f"Raw rows: {len(df_raw_ache)} -> Curated unique molecules: {len(curated_ache)}")
print("Active compounds (label=1):", (curated_ache["activity_label"] == 1).sum())
print("Inactive compounds (label=0):", (curated_ache["activity_label"] == 0).sum())
curated_ache.head()

## 7. Structural Data Mining: Torsion Preferences (CSD / PDB)

The Cambridge Structural Database (CSD) and Protein Data Bank (PDB) provide empirical crystallographic evidence for conformational preferences:

1. **C–C–C–C (Alkanes / Butane)**:
   - **Anti ($180^\circ$)** is the global energy minimum (lowest steric clash between methyl groups).
   - **Gauche ($\pm 60^\circ$)** is a local minimum, $\sim 0.9\,\text{kcal/mol}$ higher in energy.

2. **O–C–C–O (Ethylene Glycol / 1,2-Dimethoxyethane)**:
   - **Gauche ($\pm 60^\circ$)** is strongly favored over anti!
   - *Stereoelectronic explanation*: In addition to intramolecular hydrogen bonding in non-polar environments, the **gauche effect** arises from stabilizing hyperconjugative orbital interactions (donation from $\sigma_{C-H}$ bonding orbital into the adjacent $\sigma^*_{C-O}$ antibonding orbital), which is geometrically maximized at a $60^\circ$ dihedral.

3. **Ester Carbonyl Oxygen vs. Ether Oxygen as H-Bond Acceptors**:
   - In an ester ($R-\text{C}(=\text{O})-\text{O}-R'$), crystallographic surveys show that **carbonyl oxygen is a far superior hydrogen bond acceptor** compared to the ether oxygen.
   - Resonance delocalizes electron density toward the carbonyl oxygen: $R-\text{C}(=\text{O})-\text{O}-R' \longleftrightarrow R-\text{C}(\text{O}^-)=\text{O}^+-R'$. Thus, the ether oxygen has a partial positive charge, while the carbonyl oxygen carries enhanced negative charge.

In [ ]:
# Load conformational torsion scan dataset (d15_torsion_profiles.csv)
torsion_file = repo_root / "data" / "processed" / "d15_torsion_profiles.csv"
df_torsion = pd.read_csv(torsion_file)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(df_torsion["dihedral_angle_deg"], df_torsion["cccc_butane_energy_kcal_mol"], 
        label="C-C-C-C (Butane, anti preferred at 180°)", color="#1f77b4", lw=2.5)
ax.plot(df_torsion["dihedral_angle_deg"], df_torsion["occo_glycol_energy_kcal_mol"], 
        label="O-C-C-O (Glycol, gauche preferred at ±60°)", color="#d62728", lw=2.5, linestyle="--")

ax.set_title("Conformational Torsion Profiles: C-C-C-C vs O-C-C-O", fontsize=13, fontweight="bold")
ax.set_xlabel("Dihedral Angle (degrees)")
ax.set_ylabel("Relative Energy (kcal / mol)")
ax.set_xticks(range(-180, 181, 60))
ax.legend(frameon=True)
plt.tight_layout()
plt.show()

## 8. Library Property Profiling: Lipinski Physicochemical Distributions

To assess the drug-likeness of the curated AChE bioactivity library, we compute the classical **Lipinski Rule of 5** and **Veber** parameters:
- **Molecular Weight (MW)**: $\le 500\,\text{Da}$
- **Calculated LogP (MolLogP)**: $\le 5.0$
- **Hydrogen Bond Donors (HBD)**: $\le 5$
- **Hydrogen Bond Acceptors (HBA)**: $\le 10$
- **Rotatable Bonds**: $\le 10$ (Veber rule)
- **Topological Polar Surface Area (TPSA)**: $\le 140\,\text{\AA}^2$ (Veber rule)

In [ ]:
# Compute Lipinski properties for the curated AChE dataset
lipinski_records = []
for smi in curated_ache["std_smiles"]:
    m = Chem.MolFromSmiles(smi)
    props = descriptors.compute_lipinski_veber(m)
    lipinski_records.append(props)

df_props = pd.DataFrame(lipinski_records)
df_ache_full = pd.concat([curated_ache.reset_index(drop=True), df_props], axis=1)

# Plot distributions of MW, LogP, HBD, and HBA
fig, axes = plt.subplots(2, 2, figsize=(9, 6.5))

sns.histplot(df_ache_full["MW"], ax=axes[0, 0], kde=True, color="teal", bins=15)
axes[0, 0].axvline(500, color="red", ls="--", label="Ro5 limit (500)")
axes[0, 0].set_title("Molecular Weight (Da)")
axes[0, 0].legend(fontsize=9)

sns.histplot(df_ache_full["MolLogP"], ax=axes[0, 1], kde=True, color="coral", bins=15)
axes[0, 1].axvline(5.0, color="red", ls="--", label="Ro5 limit (5.0)")
axes[0, 1].set_title("Calculated LogP")
axes[0, 1].legend(fontsize=9)

sns.countplot(x=df_ache_full["HBD"], ax=axes[1, 0], color="royalblue")
axes[1, 0].set_title("H-Bond Donors (HBD)")

sns.countplot(x=df_ache_full["HBA"], ax=axes[1, 1], color="goldenrod")
axes[1, 1].set_title("H-Bond Acceptors (HBA)")

plt.tight_layout()
plt.show()

## 9. Hands-On Exercises

### Exercise 6.1: Salt Stripping and Neutralization
Write a function `clean_molecule_smiles(smiles: str) -> str` that accepts an arbitrary SMILES (which may contain salts or formal charges), strips salts retaining the largest organic fragment, neutralizes formal charges, and returns the canonical SMILES string.

In [ ]:
def clean_molecule_smiles(smiles: str) -> str:
    # Complete working solution
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = standardize.remove_salts(mol)
    mol = standardize.neutralize_charges(mol)
    return Chem.MolToSmiles(mol, isomericSmiles=True)

# Quick check
test_smi = 'CC(=O)[O-].[Na+]'
print('Result for sodium acetate:', clean_molecule_smiles(test_smi))

In [ ]:
# Self-check test for Exercise 6.1
res1 = clean_molecule_smiles("CC(=O)[O-].[Na+]")
assert res1 == "CC(=O)O", f"Expected CC(=O)O, got {res1}"

res2 = clean_molecule_smiles("CCN(CC)CC.Cl")
assert res2 == "CCN(CC)CC", f"Expected CCN(CC)CC, got {res2}"
print("✔ Exercise 6.1 passed!")

### Exercise 6.2: Duplicate Aggregation and IQR Flagging
Given a pandas DataFrame containing duplicate assay records for the same SMILES, compute the **median** $\text{IC}_{50}$ and the **Interquartile Range (IQR)**: $\text{IQR} = Q_3 - Q_1$.
Implement `aggregate_duplicates(df, smiles_col, val_col)` returning a DataFrame indexed by unique SMILES with columns `median_val` and `iqr_val`.

In [ ]:
def aggregate_duplicates(df: pd.DataFrame, smiles_col: str, val_col: str) -> pd.DataFrame:
    # Complete working solution
    def calc_iqr(x):
        q75, q25 = np.percentile(x, [75, 25])
        return q75 - q25
    
    agg_df = df.groupby(smiles_col)[val_col].agg(
        median_val="median",
        iqr_val=calc_iqr
    ).reset_index()
    return agg_df

# Quick test
toy_data = pd.DataFrame({
    'smiles': ['c1ccccc1', 'c1ccccc1', 'c1ccccc1', 'CCO', 'CCO'],
    'ic50': [10.0, 15.0, 12.0, 100.0, 120.0]
})
print(aggregate_duplicates(toy_data, 'smiles', 'ic50'))

In [ ]:
# Self-check test for Exercise 6.2
toy_df = pd.DataFrame({
    'smi': ['CCC', 'CCC', 'CCC', 'CCC'],
    'val': [10.0, 20.0, 30.0, 40.0]
})
agg_res = aggregate_duplicates(toy_df, 'smi', 'val')
assert len(agg_res) == 1
assert np.isclose(agg_res.loc[0, 'median_val'], 25.0)
assert agg_res.loc[0, 'iqr_val'] > 0
print("✔ Exercise 6.2 passed!")

### Exercise 6.3: $\text{pIC}_{50}$ Conversion & Binary Thresholding
Implement `convert_ic50_to_pic50(ic50_nm: float) -> float` according to:
$$\text{pIC}_{50} = 9 - \log_{10}(\text{IC}_{50} \text{ [nM]})$$

In [ ]:
def convert_ic50_to_pic50(ic50_nm: float) -> float:
    # Complete working solution
    if ic50_nm <= 0:
        raise ValueError("IC50 must be positive")
    return float(9.0 - np.log10(ic50_nm))

print('pIC50 for 100 nM:', convert_ic50_to_pic50(100.0))

In [ ]:
# Self-check test for Exercise 6.3
assert np.isclose(convert_ic50_to_pic50(1.0), 9.0)
assert np.isclose(convert_ic50_to_pic50(1000.0), 6.0)
assert np.isclose(convert_ic50_to_pic50(10000.0), 5.0)
print("✔ Exercise 6.3 passed!")

## 10. Common Pitfalls in Chemical Database Curation

1. **Unit Heterogeneity**: In raw literature compilations, $\text{IC}_{50}$ values may be reported in $\mu\text{M}$, $\text{nM}$, $\text{pM}$, or $\text{mg/mL}$. Never assume units without checking `standard_units`.
2. **Open-Ended Inequalities**: Assays reporting `standard_relation` as `>` or `<` indicate compounds that were insoluble, toxic, or inactive at the tested concentration. Mixing `> 10,000 nM` with an exact `= 10,000 nM` measurement ruins regression models.
3. **Conflicting Duplicate Assays**: The same compound tested in two different laboratories may show an $\text{IC}_{50}$ of $20\,\text{nM}$ in one assay and $500\,\text{nM}$ in another due to divergent substrate concentrations $[S]$, incubation times, or enzyme isoforms. Always flag or inspect duplicates with high IQR ($IQR / \text{median} > 1.0$).
4. **Salt Retention Artifacts**: Calculating molecular weight or LogP on `CC(=O)[O-].[Na+]` instead of `CC(=O)O` distorts descriptor calculations.

## 11. Summary & Key Takeaways

- **Database Diversity**: PubChem provides massive broad chemical coverage (>115M compounds); ChEMBL provides deeply curated, target-annotated pharmacology (>2.4M compounds); PDB and CSD supply experimental 3D geometry.
- **Curation Pipeline**: Standardize structures (strip salts, uncharge, canonicalize) before conducting deduplication or property calculations.
- **Logarithmic Scale**: $\text{pIC}_{50} = 9 - \log_{10}(\text{IC}_{50}\text{ [nM]})$ normalizes bioactivity to a chemically intuitive scale corresponding linearly to binding free energy $\Delta G^\circ$.
- **Conformational Preferences**: Butane strongly favors anti ($180^\circ$), whereas O–C–C–O systems favor gauche ($\pm 60^\circ$) due to stereoelectronic hyperconjugation.

## 12. References & Further Reading

- Gaulton, A. et al. The ChEMBL database in 2017. *Nucleic Acids Res.* **2017**, 45, D945–D954.
- Kim, S. et al. PubChem 2023 update. *Nucleic Acids Res.* **2023**, 51, D1373–D1380.
- Bento, A. P. et al. The ChEMBL bioactivity database: an update. *Nucleic Acids Res.* **2014**, 42, D1083–D1090.
- Fourches, D.; Muratov, E.; Tropsha, A. Trust, but verify: on the importance of chemical curation in cheminformatics and QSAR modeling. *J. Chem. Inf. Model.* **2010**, 50, 1189–1204.

In [ ]:
# Final self-test cell
print("✔ NB06 checks passed")